In [112]:
# Import the required libraries. Add more as needed, e.g. for feature selection.

import pandas as pd
import numpy as np
import re
import nltk
import statistics
from nltk.corpus import stopwords
import re
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
nltk.download('punkt')
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectKBest, chi2
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pranav_t/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/pranav_t/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Read the train and test data from csv files to dataframes

train = pd.read_csv("new_train.csv", header = None, names = ['rating', 'review'])
test = pd.read_csv("new_test.csv", header = None, names = ['review'])
print(train.head())
print(test.head())
print(train.shape)

   rating                                             review
0      -1  Eat at Fioris, they said.  Youll like it, they...
1      -1  I just don't understand the appeal.  I've trie...
2       1  This is my go to place for a really good beef ...
3      -1  Not impressed. When I ordered the Oyako bowl, ...
4      -1  This is the first time ever I wrote a bad revi...
                                              review
0  Got take-out from here last night and it was H...
1  Girls are sweet and prices are reasonable. The...
2  Rudest people I have ever encountered.  Husban...
3  This airport is only coveted for the destinati...
4  The last 4 months have shown a steady decline ...
(18000, 2)


In [3]:
# This function takes a review as an input and returns the cleaned review

def CleanText(review):

    
    # convert the text to lower case and then remove unnecessary text (numbers, spaces etc.) from the reviews using regex
    # tokenize the words
    # remove stop words
    # Apply stemming 
    
    review = review.lower()
    review = re.sub(r'[^a-zA-Z\s]', '', review)
    words = nltk.tokenize.word_tokenize(review)
    stop_words = set(stopwords.words("english"))
    cleaned_wordss = []
    for word in words:
        if word not in stop_words:
            cleaned_wordss.append(word)
    stemmer = PorterStemmer()
    cleaned_words = []
    for word in cleaned_wordss:
        cleaned_words.append(stemmer.stem(word))
    # Join the words to form a sentence and return it
    cleaned_review = " ".join(cleaned_words)
    
    return cleaned_review

In [4]:
# Apply the CleanText function to clean the reviews
train['review_cleaned'] = train['review'].apply(CleanText)
test['review_cleaned'] = test['review'].apply(CleanText)

this is training data
this is testing data


In [183]:
# This function takes all train and test reviews as input and convert them to vectors using CountVectorizer or TfidfVectorizer
def text_feature_selector(train, test):
    
    # Initialize the CountVectorizer or TF-IDF vectorizer
    # Fit and transform the cleaned reviews of train using Vectorizer
    # Transform the cleaned review of test using the Vectorizer
    # After this step you will be left with train features and test features
    
    # Next step is to select best features
    
    # Initialize feature selector
    # Fit and transform train features using the feature selector
    # Transform the test features using the feature selector
    
    # finally return the transformed train and test features
    
    # return train features, test features
    
    # try to instantiate train and test in this method
    
    train_target = train[:, 0].astype(int)
 
    
    vectorizer = TfidfVectorizer() # need to change to tfidf
    X_train = vectorizer.fit_transform(train[:, 2])
    X_test = vectorizer.transform(test[:, 2])

    classifier = SelectKBest(chi2, k = 5)
    train_features = classifier.fit_transform(X_train,train_target)
    test_features = classifier.transform(X_test)

    return train_features, test_features

In [189]:
# This class implements the KNN algorithm 

class KNN():
    def jaccard(x,y):
        return float(len(x.intersection(y)))/float(len(x.union.y))
    def cosine(x,y):
        x_np = x.values
        y_np = y.values
        return np.dot(x_np,y_np)/norm(x_np) * norm(y_np)
    
    def __init__(self, k = 9, similarity_function = 'cosine', similarity_function2 = 'jaccard'):
        
        self.k = k
        self.similarity_function = similarity_function
        self.similarity_function2 = similarity_function2
        # Try with at least two similarity functions and pass the similarity function accordingly while initializing the class object
        #euclidean distance ->
    
    # Initialize the train and target
    def fit(self, train_features, train_ratings):
        self.train_features = train_features
        self.train_ratings = train_ratings
    
    #Predict the labels for test set using the nearest neighbors in train set
    def predict(self, test):
        self.test = test
        self.most_frequent_labels = []

        # implement the similarity function and return the most frequent label of the K nearest neigbors for each sample in the test set
        # find the nieghbors and the corresponding ratings
        for test_sample in test:
            distances = [cosine_similarity(test_sample, neighbor)for neighbor in self.train_features]
            index = [i for i, _ in sorted(enumerate(distances), key=lambda x: x[1], reverse=True)[:3]]
            most_frequent_rating_labels = []
            most_frequent_rating_labels = np.array([self.train_ratings[i] for i in index])
            positive_count = np.sum(most_frequent_rating_labels == 1)
            negative_count = np.sum(most_frequent_rating_labels == -1)
            if(positive_count > negative_count):
                self.most_frequent_labels.append(1)
            else:
                self.most_frequent_labels.append(-1)
                
                
        return self.most_frequent_labels

In [180]:
# Cross validation to find the best K value. You can also do this to find the best feature set.
# First shuffle the entire train data and divide it into k roughly equal sized folds 
# The training and evaluation process is repeated 'k' times, with each iteration using a different fold as the test set and the remaining k-1 folds as the training set.
# The K-nearest neighbors classifier is used to predict the labels of the test set using the remaining (k-1) folds as train.
# The cross validation function should return the best K value (here K stands for number of nearest neighbors)

def CrossValidation(train):
    
    
    shuffled_train = train.sample(frac = 1).reset_index(drop = True)# shuffle train data (reset the indices after shuffling the data)
    k_values = [1,2,3,4,5,6,7,8,9,10]# define a list of k values
    n_folds = 3 # define the number of folds
    
    accuracies = []
    kf = KFold(n_splits = n_folds)
    for k in k_values:
        labels = []
        shuffled_train_np = shuffled_train.values
        for train_index, test_index in kf.split(shuffled_train):
            train_fold = shuffled_train_np[train_index]
            test_fold = shuffled_train_np[test_index]
            test_fold_features, train_fold_features = text_feature_selector(train_fold, test_fold)
            knn = KNN()
            knn.__init__(k = k)
            knn.fit(train_fold_features, train.iloc[:, 0])
            labels.append(knn.predict(test_fold_features))
            actual_label = test.fold.iloc[:, 0]
            accuracy = accuracy_score(actual_label, knn.predict(test_fold_features))
            accuracies.append(accuracy)
    
    best_k = k_values.index(max(accuracies))
    '''
    1) Divide the data into k folds (n_folds)
    
    2) After dividing the data into k folds,for each fold call text_feature_selector() to get the features of train folds and test fold
    
    3) Initialize and fit the KNN class and with the help of predict method in the KNN class, get the labels for the test fold
    
    4) Each example in test fold has a target (rating), with the help of predicted label and target, find the accuracy for each fold
    
    5) Find the average accuracy of all k-folds
    
    Repeat the above steps for all values of k in k_values
    
    Find the value of k for which the average accuracy of all folds is high compared to other k values and return it
    Also return the average accuracy of k-folds for each value of k
    '''
    
    
    return best_k, accuracies

In [181]:
best_k, accuracies = CrossValidation(train)

KeyboardInterrupt: 

In [ ]:
# Plot the accuracies (y-axis) vs k_values (x-axis) using matplotlib
plt.plot(k_values, accuracies, marker='o')
plt.title('Accuracy vs K Values')
plt.xlabel('K Values')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

In [ ]:
# Initialize the KNN class with best_k value and predict the labels for test set using the entire train set
train_np = np.array(train)
test_np = np.array(test)
train1, test1 = text_feature_selector(train_np, test_np)
knn = KNN()
knn.fit(train1, test1)
predicted_labels = knn.predict(test1)

/var/folders/c6/kg2jf71x0l9fs4wvzq9b58_40000gn/T/ipykernel_17627/808198782.py:36: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  positive_count = np.sum(most_frequent_rating_labels == 1)
/var/folders/c6/kg2jf71x0l9fs4wvzq9b58_40000gn/T/ipykernel_17627/808198782.py:37: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  negative_count = np.sum(most_frequent_rating_labels == -1)


In [ ]:
# Write the predicted labels of the test set to a .txt file and upload the .txt file to miner
with open('predicted_labels.txt', 'w') as file:
    for label in predicted_labels:
        file.write(f"{label}\n")